# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [11]:
import pandas as pd
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

x = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id'],
)
print(x)


           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07


One row of my result is one: Each row will have item, quanitity of the item, order number, and then vendor ID. There will be 4 rows.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [22]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
normal = pd.json_normalize(
    orders_json,
    record_path = 'lines',
    meta=['order', 'vendor_id'],
)
print(normal)
# TODO: assert the row count and the quantity total
assert len(normal) ==4
print('check #1 passed')
assert normal['qty'].sum() == 7
print('check #2 passed')

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07
check #1 passed
check #2 passed


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [25]:
# TODO
joined = pd.merge(
    normal,
    pd.json_normalize(vendors_json),
    on='vendor_id',
    how='left',
    indicator=True,
)
print(joined)
print("V-07 did not match and there are no units attached to it")

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only
V-07 did not match and there are no units attached to it


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [28]:
# TODO
joined.groupby('name')['qty'].sum()
#I dropped Vendor 07 because it has no values to it. If it had a zero value to it, I would have kept it in.

,qty
name,
Cav Merch,1
Hoos Burgers,3
NaN,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [29]:
# TODO
joined.groupby('zone')['qty'].sum()
#Zone A sold the most items

,qty
zone,
A,4


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [31]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]
pd.json_normalize(ragged_json)


,order,vendor_id,lines
0,4,V-01,"[{'item': 'Soda', 'qty': 2}]"
1,5,V-10,NaN
2,6,V-01,[{'item': 'Fries'}]


### Q6 — Validate

In [32]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

NameError: name 'lines' is not defined

### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

_your answer here_